# Shared Test Evaluation — All 14 Experiments

Loads best models from both experiment notebooks and produces a unified comparison
across all 14 experiments, with test metrics reported separately for:
- **Local** test scenes (1 scene)
- **Mukherjee** test scenes (9 scenes, SID78 pinned)
- **Combined** (all 10 scenes)

Run this notebook after both training notebooks have completed.

| Exps | Notebook | Training data |
|------|----------|---------------|
| 1–6  | `unet_finetune_mullen_colab.ipynb` | Local NC only |
| 7–14 | `unet_finetune_mullen_mukherjee_colab.ipynb` | Mukherjee ± local |

---
## 0.1 Install & Mount

In [1]:
import subprocess, sys
pkgs = ["rasterio", "albumentations", "torchmetrics>=1.3",
        "segmentation-models-pytorch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("Ready")

Mounted at /content/drive
Ready


---
## 0.2 Configuration

In [8]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import random, warnings, torch
import torch.nn as nn
import segmentation_models_pytorch as smp
import torchmetrics
from torchmetrics.functional import precision_recall_curve
import rasterio
from rasterio.windows import Window
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
warnings.filterwarnings("ignore")

DRIVE_ROOT      = Path("/content/drive/MyDrive/Research/EO-Methane/paper2")
EXP_LOCAL_DIR   = DRIVE_ROOT / "experiments_mullen"      # Exps 1-6
EXP_MUK_DIR     = DRIVE_ROOT / "experiments_mullen_muk"  # Exps 7-14
OUTPUT_DIR      = DRIVE_ROOT / "shared_eval"
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Data paths ────────────────────────────────────────────────────────────────
# ── Local NC data (same as Experiments 1-6) ───────────────────────────────────
LOCAL_IMAGE_DIR    = DRIVE_ROOT / "data" / "local_nc" / "20240716_151643_20_24c8" / "PS"
LOCAL_SLOPE_DIR    = DRIVE_ROOT / "data" / "local_nc" / "slope"
LOCAL_MASK_DIR     = DRIVE_ROOT / "data" / "local_nc" / "20240716_151643_20_24c8" / "labels"
# ── Mukherjee data ────────────────────────────────────────────────────────────
MUK_IMAGE_DIR   = DRIVE_ROOT / "data" / "Mukherjee_2024" / "PS"
MUK_SLOPE_DIR   = DRIVE_ROOT / "data" / "Mukherjee_2024" / "slope"
MUK_MASK_DIR    = DRIVE_ROOT / "data" / "Mukherjee_2024" / "labels_binary"

CHIP_SIZE  = 256
BATCH_SIZE = 8
RANDOM_SEED= 42

# Band / normalisation constants
BAND_REORDER      = [1, 2, 3, 5, 6, 7, 8, 9, 0, 4]
N_BANDS_SLOPE     = 11
N_BANDS_NOSLOPE   = 10
EFFICIENTNET_MEAN = 0.449
EFFICIENTNET_STD  = 0.226

# Split params (must match training notebooks)
LOCAL_N_VAL  = 2
LOCAL_N_TEST = 1
MUK_VAL_FRAC = 0.10
MUK_TEST_FRAC= 0.10
MUK_PINNED   = ["SID78"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED)
print(f"Device: {DEVICE}")

Device: cuda


---
## 1. Shared Utilities

In [9]:
from dataclasses import dataclass

@dataclass
class Scene:
    img_path: Path; slope_path: Path; mask_path: Path; source: str

def reorder_bands(chip): return chip[BAND_REORDER, :, :]

def preprocess_spectral(chip):
    chip = chip.astype(np.float32)
    chip = (chip / 10000.0) * 255.0
    chip[chip == 0] = 0.5
    chip = chip / 255.0
    return (chip - EFFICIENTNET_MEAN) / EFFICIENTNET_STD

def preprocess_slope(slope, mean, std):
    slope = slope.astype(np.float32)
    slope[slope == -9999] = np.nan
    slope = np.nan_to_num(slope, nan=0.0)
    return (slope - mean) / (std + 1e-6)

def get_chip_offsets(total, chip_size):
    offsets = list(range(0, total - chip_size + 1, chip_size))
    if not offsets or offsets[-1] + chip_size < total:
        offsets.append(max(0, total - chip_size))
    return offsets

def build_model(n_bands):
    return smp.Unet(encoder_name="efficientnet-b7", encoder_weights=None,
                    in_channels=n_bands, classes=2, activation=None)

def make_metrics():
    return {k: cls(task="binary", ignore_index=255).to(DEVICE)
            for k, cls in [("iou",torchmetrics.JaccardIndex),
                           ("f1",torchmetrics.F1Score),
                           ("precision",torchmetrics.Precision),
                           ("recall",torchmetrics.Recall)]}

class TestDataset(Dataset):
    def __init__(self, scenes, chip_index, chip_size, slope_stats, n_bands):
        self.scenes = scenes; self.chip_index = chip_index
        self.chip_size = chip_size; self.slope_stats = slope_stats
        self.n_bands = n_bands
    def __len__(self): return len(self.chip_index)
    def _read(self, path, row, col, bands=None):
        window = Window(col, row, self.chip_size, self.chip_size)
        with rasterio.open(path) as src:
            return src.read([b+1 for b in bands] if bands else None,
                            window=window, out_dtype=np.float32)
    def __getitem__(self, idx):
        scene_i, row, col = self.chip_index[idx]
        s = self.scenes[scene_i]
        sm, ss = self.slope_stats[s.source]
        spec  = preprocess_spectral(self._read(s.img_path, row, col))
        if self.n_bands == N_BANDS_SLOPE:
            slp   = preprocess_slope(self._read(s.slope_path, row, col), sm, ss)
            image = np.concatenate([spec, slp], axis=0)
        else:
            image = spec
        image = np.nan_to_num(image, nan=0.0, posinf=3.0, neginf=-3.0)
        mask  = self._read(s.mask_path, row, col)[0].astype(np.uint8)
        return torch.from_numpy(image), torch.from_numpy(mask.astype(np.int64))

def build_chip_index(scenes, chip_size=256):
    chips = []
    for i, s in enumerate(scenes):
        with rasterio.open(s.img_path) as src: H, W = src.height, src.width
        for r in get_chip_offsets(H, chip_size):
            for c in get_chip_offsets(W, chip_size):
                chips.append((i, r, c))
    return chips

def make_loader(scenes, n_bands, slope_stats):
    chips = build_chip_index(scenes)
    ds    = TestDataset(scenes, chips, CHIP_SIZE, slope_stats, n_bands)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)

class DiceLoss(nn.Module):
    def __init__(self): super().__init__()
    def forward(self, logits, targets):
        valid = targets != 255
        p = torch.softmax(logits,dim=1)[:,1][valid]; t=targets[valid].float()
        return 1.-(2.*(p*t).sum()+1)/(p.sum()+t.sum()+1)

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss(ignore_index=255)
        self.dice = DiceLoss()
    def forward(self, logits, targets):
        return 0.5*self.ce(logits,targets)+0.5*self.dice(logits,targets)

criterion = CombinedLoss()

def eval_loader(model, loader, threshold=None):
    m = make_metrics(); model.eval(); total_loss=0.
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            total_loss += criterion(logits,masks).item()
            if threshold is None:
                preds = logits.argmax(dim=1)
            else:
                preds = (torch.softmax(logits,dim=1)[:,1]>=threshold).long()
            for v in m.values(): v.update(preds,masks)
    return {"loss":total_loss/len(loader),
            "iou":m["iou"].compute().item(),
            "f1":m["f1"].compute().item(),
            "precision":m["precision"].compute().item(),
            "recall":m["recall"].compute().item()}

def find_opt_thresh(model, loader):
    model.eval(); ap,al=[],[]
    with torch.no_grad():
        for images, masks in loader:
            images,masks=images.to(DEVICE),masks.to(DEVICE)
            probs=torch.softmax(model(images),dim=1)[:,1]
            valid=masks!=255
            ap.append(probs[valid].cpu()); al.append(masks[valid].cpu())
    ap=torch.cat(ap); al=torch.cat(al)
    prec,rec,thresholds=precision_recall_curve(ap,al,task="binary")
    f1s=2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-6)
    return float(thresholds[f1s.argmax()])

print("Utilities defined.")

Utilities defined.


In [18]:
sorted(LOCAL_IMAGE_DIR.glob("*.tif"))

[PosixPath('/content/drive/MyDrive/Research/EO-Methane/paper2/data/local_nc/20240716_151643_20_24c8/PS/tile_10240_6144.tif'),
 PosixPath('/content/drive/MyDrive/Research/EO-Methane/paper2/data/local_nc/20240716_151643_20_24c8/PS/tile_10240_7168.tif'),
 PosixPath('/content/drive/MyDrive/Research/EO-Methane/paper2/data/local_nc/20240716_151643_20_24c8/PS/tile_11264_6144.tif'),
 PosixPath('/content/drive/MyDrive/Research/EO-Methane/paper2/data/local_nc/20240716_151643_20_24c8/PS/tile_11264_7168.tif'),
 PosixPath('/content/drive/MyDrive/Research/EO-Methane/paper2/data/local_nc/20240716_151643_20_24c8/PS/tile_11264_8192.tif'),
 PosixPath('/content/drive/MyDrive/Research/EO-Methane/paper2/data/local_nc/20240716_151643_20_24c8/PS/tile_8192_6144.tif'),
 PosixPath('/content/drive/MyDrive/Research/EO-Methane/paper2/data/local_nc/20240716_151643_20_24c8/PS/tile_8192_7168.tif'),
 PosixPath('/content/drive/MyDrive/Research/EO-Methane/paper2/data/local_nc/20240716_151643_20_24c8/PS/tile_9216_6144.ti

In [22]:
LOCAL_MASK_DIR/sorted(LOCAL_MASK_DIR.glob("*.tif"))[0].name.replace('tile', 'label')

PosixPath('/content/drive/MyDrive/Research/EO-Methane/paper2/data/local_nc/20240716_151643_20_24c8/labels/label_10240_6144.tif')

---
## 2. Reconstruct Scene Splits & Load Slope Stats

In [23]:
def load_scenes(image_dir, slope_dir, mask_dir, source):
    scenes = []
    for img in sorted(image_dir.glob("*.tif")):
        if source == 'local':
            sp=slope_dir/img.name; mp=mask_dir/img.name.replace('tile', 'label')
        else:
            sp=slope_dir/img.name; mp=mask_dir/img.name
        if sp.exists() and mp.exists():
            scenes.append(Scene(img,sp,mp,source))
    print(f"  [{source}] {len(scenes)} scenes")
    return scenes

def find_scene(scenes, stem):
    for i,s in enumerate(scenes):
        if stem.lower() in s.img_path.stem.lower(): return i
    raise ValueError(f"{stem} not found")

print("Loading scenes...")
local_scenes = load_scenes(LOCAL_IMAGE_DIR, LOCAL_SLOPE_DIR, LOCAL_MASK_DIR, "local")
muk_scenes   = load_scenes(MUK_IMAGE_DIR,   MUK_SLOPE_DIR,   MUK_MASK_DIR,  "mukherjee")

# Reproduce local split
random.seed(RANDOM_SEED)
lo = list(range(len(local_scenes))); random.shuffle(lo)
local_test_idx = set(lo[:LOCAL_N_TEST])

# Reproduce Mukherjee split
n_muk      = len(muk_scenes)
n_muk_test = max(1, round(n_muk*MUK_TEST_FRAC))
pinned     = {find_scene(muk_scenes, s) for s in MUK_PINNED}
remaining  = [i for i in range(n_muk) if i not in pinned]
random.shuffle(remaining)
muk_test_idx = pinned | set(remaining[:n_muk_test-len(pinned)])

local_test_scenes = [local_scenes[i] for i in sorted(local_test_idx)]
muk_test_scenes   = [muk_scenes[i]   for i in sorted(muk_test_idx)]
combined_test     = local_test_scenes + muk_test_scenes

print(f"Test — local:{len(local_test_scenes)}  "
      f"mukherjee:{len(muk_test_scenes)}  combined:{len(combined_test)}")
print("Local test:",   [s.img_path.name for s in local_test_scenes])
print("Mukherjee test:",[s.img_path.name for s in muk_test_scenes])

# Load slope stats saved by mukherjee notebook
slope_stats_path = EXP_MUK_DIR / "slope_stats.npy"
if slope_stats_path.exists():
    ss = np.load(slope_stats_path, allow_pickle=True).item()
    LOCAL_SLOPE_MEAN = ss["local_mean"]; LOCAL_SLOPE_STD = ss["local_std"]
    MUK_SLOPE_MEAN   = ss["muk_mean"];   MUK_SLOPE_STD   = ss["muk_std"]
    print(f"Slope stats — local: {LOCAL_SLOPE_MEAN:.2f}±{LOCAL_SLOPE_STD:.2f}  "
          f"muk: {MUK_SLOPE_MEAN:.2f}±{MUK_SLOPE_STD:.2f}")
else:
    raise FileNotFoundError(
        "slope_stats.npy not found. Run unet_finetune_mullen_mukherjee_colab.ipynb first.")

SLOPE_STATS = {
    "local":     (LOCAL_SLOPE_MEAN, LOCAL_SLOPE_STD),
    "mukherjee": (MUK_SLOPE_MEAN,   MUK_SLOPE_STD),
}

Loading scenes...
  [local] 9 scenes
  [mukherjee] 81 scenes
Test — local:1  mukherjee:8  combined:9
Local test: ['tile_11264_7168.tif']
Mukherjee test: ['SID21.tif', 'SID33.tif', 'SID40.tif', 'SID43.tif', 'SID56.tif', 'SID69.tif', 'SID73.tif', 'SID78.tif']
Slope stats — local: 4.60±5.33  muk: 8.69±10.93


---
## 3. Experiment Registry
Maps every experiment to its checkpoint directory, band count, and test split.

In [24]:
# (exp_number, name, ckpt_dir, n_bands, short_label)
EXPERIMENTS = [
    # Experiments 1-6 (local only)
    (1,  "mullen_unaltered",                   EXP_LOCAL_DIR, 5,  "+slope unalt."),
    (2,  "mullen_noslope_unaltered",            EXP_LOCAL_DIR, 4,  "-slope unalt."),
    (3,  "mullen_finetune_standard",            EXP_LOCAL_DIR, 11, "+slope std"),
    (4,  "mullen_finetune_aggressive",          EXP_LOCAL_DIR, 11, "+slope agg"),
    (5,  "mullen_noslope_finetune_standard",    EXP_LOCAL_DIR, 10, "-slope std"),
    (6,  "mullen_noslope_finetune_aggressive",  EXP_LOCAL_DIR, 10, "-slope agg"),
    # Experiments 7-14 (mukherjee ± local)
    (7,  "mullen_muk_slope_standard",           EXP_MUK_DIR,   11, "muk+slope std"),
    (8,  "mullen_muk_slope_aggressive",         EXP_MUK_DIR,   11, "muk+slope agg"),
    (9,  "mullen_muk_noslope_standard",         EXP_MUK_DIR,   10, "muk-slope std"),
    (10, "mullen_muk_noslope_aggressive",       EXP_MUK_DIR,   10, "muk-slope agg"),
    (11, "mullen_muk_local_slope_standard",     EXP_MUK_DIR,   11, "muk+loc+slope std"),
    (12, "mullen_muk_local_slope_aggressive",   EXP_MUK_DIR,   11, "muk+loc+slope agg"),
    (13, "mullen_muk_local_noslope_standard",   EXP_MUK_DIR,   10, "muk+loc-slope std"),
    (14, "mullen_muk_local_noslope_aggressive", EXP_MUK_DIR,   10, "muk+loc-slope agg"),
]

print(f"Registry: {len(EXPERIMENTS)} experiments")

Registry: 14 experiments


---
## 4. Evaluate All Experiments

In [25]:
all_results = []   # list of dicts for DataFrame

for exp_num, name, ckpt_dir, n_bands, short_label in EXPERIMENTS:
    model_path = ckpt_dir / name / "best_model.pt"
    if not model_path.exists():
        print(f"  Exp {exp_num} ({name}): best_model.pt not found — skipping")
        continue

    print(f"\n── Exp {exp_num}: {name} ({n_bands}-band) ──")
    model = build_model(n_bands).to(DEVICE)
    model.load_state_dict(
        torch.load(model_path, map_location=DEVICE, weights_only=True))

    # Build test loaders
    local_loader    = make_loader(local_test_scenes,  n_bands, SLOPE_STATS)
    muk_loader      = make_loader(muk_test_scenes,    n_bands, SLOPE_STATS)
    combined_loader = make_loader(combined_test,      n_bands, SLOPE_STATS)

    # Find optimal threshold on combined test set
    opt_thresh = find_opt_thresh(model, combined_loader)
    print(f"  Optimal threshold: {opt_thresh:.3f}")

    for split_name, loader in [("local",    local_loader),
                               ("mukherjee",muk_loader),
                               ("combined", combined_loader)]:
        r05  = eval_loader(model, loader, threshold=None)
        ropt = eval_loader(model, loader, threshold=opt_thresh)
        print(f"  [{split_name}] @0.50: IoU={r05['iou']:.4f} F1={r05['f1']:.4f} "
              f"P={r05['precision']:.4f} R={r05['recall']:.4f}")
        print(f"  [{split_name}] @{opt_thresh:.3f}: IoU={ropt['iou']:.4f} "
              f"F1={ropt['f1']:.4f} P={ropt['precision']:.4f} R={ropt['recall']:.4f}")
        for thresh_label, res in [("0.50", r05), ("opt", ropt)]:
            all_results.append({
                "exp":        exp_num,
                "name":       name,
                "short":      short_label,
                "n_bands":    n_bands,
                "test_split": split_name,
                "threshold":  thresh_label if thresh_label=="0.50"
                              else round(opt_thresh,3),
                "iou":        round(res["iou"],4),
                "f1":         round(res["f1"],4),
                "precision":  round(res["precision"],4),
                "recall":     round(res["recall"],4),
            })

df_all = pd.DataFrame(all_results)
df_all.to_csv(OUTPUT_DIR / "all_experiments_results.csv", index=False)
print(f"\nSaved: all_experiments_results.csv")
display(df_all[df_all["threshold"]=="0.50"].pivot_table(
    index=["exp","short"], columns="test_split",
    values="f1", aggfunc="first").round(4))


── Exp 1: mullen_unaltered (5-band) ──


RuntimeError: Given groups=1, weight of size [64, 5, 3, 3], expected input[8, 10, 257, 257] to have 5 channels, but got 10 channels instead

---
## 5. Comparison Plots

In [ ]:
# ── F1 at 0.5 threshold — all experiments, 3 test splits ─────────────────────
df_05 = df_all[df_all["threshold"]=="0.50"].copy()
exp_order = sorted(df_05["exp"].unique())
splits    = ["local","mukherjee","combined"]
split_colours = {"local":"#4C72B0", "mukherjee":"#55A868", "combined":"#C44E52"}

x      = np.arange(len(exp_order))
width  = 0.25
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
axes = axes.flatten()

for ax_i, metric in enumerate(["f1","iou","precision","recall"]):
    ax = axes[ax_i]
    for j, split in enumerate(splits):
        sub = df_05[df_05["test_split"]==split].set_index("exp")
        vals = [sub.loc[e, metric] if e in sub.index else 0 for e in exp_order]
        bars = ax.bar(x + j*width, vals, width, label=split,
                      color=split_colours[split], alpha=0.85)
        for bar, val in zip(bars, vals):
            if val > 0:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                        f"{val:.3f}", ha="center", va="bottom", fontsize=5.5,
                        rotation=90)
    short_labels = [df_05[df_05["exp"]==e]["short"].iloc[0]
                    if len(df_05[df_05["exp"]==e]) else str(e)
                    for e in exp_order]
    ax.set_xticks(x + width)
    ax.set_xticklabels([f"{n}\n{l}" for n,l in zip(exp_order, short_labels)],
                       fontsize=6)
    ax.set_ylim(0, 1.18); ax.set_ylabel(metric.upper())
    ax.set_title(metric.upper(), fontsize=11, fontweight="bold")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
    # Vertical line between Exp 6 and 7
    if 6 in exp_order and 7 in exp_order:
        mid = (exp_order.index(6) + exp_order.index(7)) / 2
        ax.axvline(x=mid + width, color="grey", linestyle="--",
                   alpha=0.5, linewidth=1.2)
        ax.text(mid + width, 1.14, "local only | +Mukherjee",
                ha="center", fontsize=6, color="grey")

plt.suptitle("All 14 Experiments — Test Metrics at 0.50 Threshold",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"all_experiments_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Default vs optimal threshold comparison — F1 only, combined test split ────
df_comb = df_all[df_all["test_split"]=="combined"].copy()
df_05c  = df_comb[df_comb["threshold"]=="0.50"].set_index("exp")
df_optc = df_comb[df_comb["threshold"]!="0.50"].set_index("exp")

x, width = np.arange(len(exp_order)), 0.35
fig, ax  = plt.subplots(figsize=(16, 5))
for i, e in enumerate(exp_order):
    v05  = df_05c.loc[e,  "f1"] if e in df_05c.index  else 0
    vopt = df_optc.loc[e, "f1"] if e in df_optc.index else 0
    thresh_val = df_optc.loc[e, "threshold"] if e in df_optc.index else ""
    b1 = ax.bar(i-width/2, v05,  width, color="#AED6F1", edgecolor="grey", alpha=0.9)
    b2 = ax.bar(i+width/2, vopt, width, color="#1A5276", edgecolor="grey", alpha=0.9)
    ax.text(b1[0].get_x()+b1[0].get_width()/2, v05+0.005,
            f"{v05:.3f}", ha="center", va="bottom", fontsize=7)
    ax.text(b2[0].get_x()+b2[0].get_width()/2, vopt+0.005,
            f"{vopt:.3f}\n@{thresh_val}",
            ha="center", va="bottom", fontsize=6)

short_labels = [df_comb[df_comb["exp"]==e]["short"].iloc[0]
                if len(df_comb[df_comb["exp"]==e]) else str(e)
                for e in exp_order]
ax.set_xticks(range(len(exp_order)))
ax.set_xticklabels([f"{n}: {l}" for n,l in zip(exp_order,short_labels)],
                   fontsize=7, rotation=15, ha="right")
ax.set_ylim(0, 1.15); ax.set_ylabel("F1 Score")
ax.set_title("F1 Score — Default (0.50) vs Optimal Threshold\nCombined Test Set",
             fontsize=11)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#AED6F1",label="thresh=0.50"),
                   Patch(color="#1A5276",label="thresh=optimal")],
          fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"threshold_comparison_all.png", dpi=150)
plt.show()

In [ ]:
# ── Split breakdown heatmap — F1 at 0.5 threshold ────────────────────────────
import matplotlib
pivot = df_05[["exp","short","test_split","f1"]].pivot_table(
    index=["exp","short"], columns="test_split", values="f1", aggfunc="first"
).reindex(columns=["local","mukherjee","combined"])

fig, ax = plt.subplots(figsize=(8, len(pivot)*0.45+1.5))
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks([0,1,2]); ax.set_xticklabels(["Local","Mukherjee","Combined"],
                                              fontsize=10)
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels([f"{i+1}: {s}" for i,(e,s) in enumerate(pivot.index)], fontsize=8)
for r in range(len(pivot)):
    for c in range(3):
        val = pivot.values[r,c]
        if not np.isnan(val):
            ax.text(c, r, f"{val:.3f}", ha="center", va="center",
                    fontsize=8, color="black" if 0.3<val<0.7 else "white")
plt.colorbar(im, ax=ax, label="F1 Score")
ax.set_title("F1 Score Heatmap — All Experiments × Test Splits\n(threshold=0.50)",
             fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"f1_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Plots at Optimal Threshold

Repeats the 4-panel bar chart and F1 heatmap using each experiment's optimal
threshold rather than the default 0.50. Experiments where the optimal threshold
is close to 0.50 will look similar; those with high optimal thresholds (e.g. the
unaltered Mullen models) may show meaningfully different precision/recall balance.

In [ ]:
# ── 4-panel bar chart at optimal threshold ────────────────────────────────────
# Optimal threshold rows have numeric (float) threshold values, not "0.50"
df_opt = df_all[df_all["threshold"] != "0.50"].copy()

fig, axes = plt.subplots(2, 2, figsize=(18, 11))
axes = axes.flatten()

for ax_i, metric in enumerate(["f1", "iou", "precision", "recall"]):
    ax = axes[ax_i]
    for j, split in enumerate(splits):
        sub  = df_opt[df_opt["test_split"] == split].set_index("exp")
        vals = [sub.loc[e, metric] if e in sub.index else 0 for e in exp_order]
        bars = ax.bar(x + j*width, vals, width, label=split,
                      color=split_colours[split], alpha=0.85)
        for bar, val, e in zip(bars, vals, exp_order):
            if val > 0:
                # Annotate with value and threshold
                thresh = sub.loc[e, "threshold"] if e in sub.index else ""
                ax.text(bar.get_x()+bar.get_width()/2,
                        bar.get_height()+0.005,
                        f"{val:.3f}\n@{thresh}",
                        ha="center", va="bottom", fontsize=4.5, rotation=90)

    short_labels = [df_opt[df_opt["exp"]==e]["short"].iloc[0]
                    if len(df_opt[df_opt["exp"]==e]) else str(e)
                    for e in exp_order]
    ax.set_xticks(x + width)
    ax.set_xticklabels([f"{n}\n{l}" for n,l in zip(exp_order, short_labels)],
                       fontsize=6)
    ax.set_ylim(0, 1.22); ax.set_ylabel(metric.upper())
    ax.set_title(metric.upper(), fontsize=11, fontweight="bold")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
    if 6 in exp_order and 7 in exp_order:
        mid = (exp_order.index(6) + exp_order.index(7)) / 2
        ax.axvline(x=mid + width, color="grey", linestyle="--",
                   alpha=0.5, linewidth=1.2)
        ax.text(mid + width, 1.18, "local only | +Mukherjee",
                ha="center", fontsize=6, color="grey")

plt.suptitle("All 14 Experiments — Test Metrics at Optimal Threshold",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"all_experiments_comparison_opt_thresh.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── F1 heatmap at optimal threshold ──────────────────────────────────────────
pivot_opt = df_opt[["exp","short","test_split","f1"]].pivot_table(
    index=["exp","short"], columns="test_split",
    values="f1", aggfunc="first"
).reindex(columns=["local","mukherjee","combined"])

fig, axes = plt.subplots(1, 2, figsize=(16, len(pivot_opt)*0.45+2))

for ax, pivot, title in [
    (axes[0], pivot.values,     "F1 @ 0.50 threshold"),
    (axes[1], pivot_opt.values, "F1 @ Optimal threshold"),
]:
    im = ax.imshow(pivot, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks([0,1,2])
    ax.set_xticklabels(["Local","Mukherjee","Combined"], fontsize=10)
    ax.set_yticks(range(len(pivot_opt)))
    ax.set_yticklabels([f"{e}: {s}" for e,s in pivot_opt.index], fontsize=8)
    for r in range(len(pivot_opt)):
        for c in range(3):
            val = pivot[r, c]
            if not np.isnan(val):
                colour = "black" if 0.3 < val < 0.7 else "white"
                ax.text(c, r, f"{val:.3f}", ha="center", va="center",
                        fontsize=8, color=colour)
    ax.set_title(title, fontsize=11, fontweight="bold")
    plt.colorbar(im, ax=ax, label="F1 Score", fraction=0.03)

plt.suptitle("F1 Score Heatmap — Default vs Optimal Threshold",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"f1_heatmap_both_thresholds.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Threshold gain table — how much does optimal threshold help? ──────────────
# Shows the F1 gain from using the optimal threshold vs 0.50 for each
# experiment and test split. Large gains indicate the model is poorly
# calibrated at 0.50 (common for unaltered/zero-shot models).
df_05_idx  = df_all[df_all["threshold"]=="0.50"].set_index(["exp","test_split"])
df_opt_idx = df_all[df_all["threshold"]!="0.50"].set_index(["exp","test_split"])

gain_rows = []
for exp_num, name, _, n_bands, short_label in EXPERIMENTS:
    for split in ["local","mukherjee","combined"]:
        key = (exp_num, split)
        if key not in df_05_idx.index or key not in df_opt_idx.index:
            continue
        f1_05  = df_05_idx.loc[key,  "f1"]
        f1_opt = df_opt_idx.loc[key, "f1"]
        thresh = df_opt_idx.loc[key, "threshold"]
        gain_rows.append({
            "exp":       exp_num,
            "short":     short_label,
            "split":     split,
            "f1_050":    round(float(f1_05),  4),
            "f1_opt":    round(float(f1_opt), 4),
            "threshold": thresh,
            "f1_gain":   round(float(f1_opt) - float(f1_05), 4),
        })

df_gain = pd.DataFrame(gain_rows)
display(df_gain.pivot_table(
    index=["exp","short"], columns="split",
    values="f1_gain", aggfunc="first"
).reindex(columns=["local","mukherjee","combined"]).round(4))
df_gain.to_csv(OUTPUT_DIR/"threshold_gain.csv", index=False)
print("\nNote: large gains (>0.05) indicate the model benefits substantially "
      "from threshold calibration.")
print(f"Saved: threshold_gain.csv")